# Thay ô trống bằng chuỗi `text`

Notebook này đọc file CSV repository GitHub và thay các ô trống trong hai cột:
- `languages`
- `primary_language`

bằng chuỗi ký tự **`text`**.

Các cột khác và những giá trị đã có trong hai cột trên sẽ được giữ nguyên. File gốc không bị ghi đè; kết quả được lưu thành file mới.

## 1. Đọc file CSV

Đặt `github_repos_complete.csv` cùng thư mục với notebook, hoặc sửa đường dẫn `INPUT_FILE` cho phù hợp.

In [2]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("github_repos_complete_topics_filled.csv")
OUTPUT_FILE = Path("github_repos_text_filled.csv")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file: {INPUT_FILE.resolve()}\n"
        "Hãy đặt CSV cạnh notebook hoặc sửa INPUT_FILE."
    )

df = pd.read_csv(INPUT_FILE, low_memory=False)

print(f"Số dòng: {len(df):,}")
print("Các cột:", df.columns.tolist())

Số dòng: 12,000
Các cột: ['repo_id', 'owner', 'repo_name', 'full_name', 'primary_language', 'languages', 'description', 'read_me', 'topics', 'license_key', 'created_at', 'updated_at', 'pushed_at', 'collected_at', 'star_count', 'forks_count', 'open_issues_count', 'size_kb', 'has_issues', 'has_wiki', 'default_branch']


## 2. Kiểm tra cột và thay ô trống bằng `text`

Code xem cả giá trị thiếu (`NaN`/`None`) lẫn chuỗi rỗng hoặc chỉ có khoảng trắng là ô trống.

In [3]:
TARGET_COLUMNS = ["languages", "primary_language"]
missing_columns = set(TARGET_COLUMNS) - set(df.columns)

if missing_columns:
    raise ValueError(f"CSV thiếu các cột bắt buộc: {missing_columns}")

def is_blank(series):
    # Ô trống gồm giá trị NA, chuỗi rỗng, hoặc chỉ có khoảng trắng
    return series.isna() | series.astype("string").str.strip().eq("").fillna(False)

for column in TARGET_COLUMNS:
    blank_mask = is_blank(df[column])
    number_filled = int(blank_mask.sum())

    # Chỉ thay các ô trống, không thay đổi giá trị đã có
    df.loc[blank_mask, column] = "text"

    print(f"{column}: đã thay {number_filled:,} ô trống bằng 'text'")

languages: đã thay 0 ô trống bằng 'text'
primary_language: đã thay 958 ô trống bằng 'text'


## 3. Lưu file kết quả

Kết quả được lưu thành `github_repos_text_filled.csv`; file gốc vẫn nguyên vẹn.

In [4]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {OUTPUT_FILE.resolve()}")

Đã lưu: D:\HomeWork\IE313\Đồ án\analysis-popularity-github-repository\notebooks\data\raw\github_repos_text_filled.csv


## 4. Xác nhận kết quả

Đọc lại file đầu ra và kiểm tra xem hai cột còn ô trống hay không.

In [5]:
check_df = pd.read_csv(OUTPUT_FILE, low_memory=False)

for column in TARGET_COLUMNS:
    remaining_blank = int(is_blank(check_df[column]).sum())
    print(f"{column}: số ô trống còn lại = {remaining_blank}")

display(check_df[TARGET_COLUMNS].head(10))

languages: số ô trống còn lại = 0
primary_language: số ô trống còn lại = 0


,languages,primary_language
0,"[""Python"", ""Svelte"", ""JavaScript"", ""TypeScript...",Python
1,"[""Rust"", ""Python"", ""Shell"", ""Dockerfile"", ""Bat...",Rust
2,"[""Python"", ""JavaScript"", ""CSS"", ""Jinja"", ""Shell""]",Python
3,"[""Java"", ""Python""]",Java
4,"[""Python"", ""Shell""]",Python
5,"[""Python"", ""TypeScript"", ""Dockerfile"", ""Jupyte...",Python
6,"[""Python"", ""GLSL"", ""HTML""]",Python
7,[],text
8,"[""TypeScript"", ""Vue"", ""SCSS"", ""JavaScript"", ""D...",TypeScript
9,"[""Python"", ""Shell""]",Python


**Lưu ý:** `text` ở đây là chuỗi ký tự thông thường, không phải giá trị ngôn ngữ thực tế. Nếu sau này phân tích thống kê hoặc xây dựng mô hình, hãy cân nhắc xử lý các giá trị `text` này như dữ liệu thiếu được đánh dấu, thay vì xem chúng là tên ngôn ngữ.